# SVD / MAB 모델 평가 (가윤 담당)

`docs/recommendation_models.md`에 정의된 공통 인터페이스(`recommend(user, ctx) -> pd.Series`)로
논문(이한우, 「게임 분류와 MAB를 이용한 게임 추천 시스템」, 서강대, 2022)의 **SVD**와 **MAB
(Multi-Armed Bandit)** 를 구현하고, 현빈이 만든 4개 모델(popularity/content/user_cf/item_cf)과
같은 기준(NDCG@10, hold-out 30%, seed=42)으로 비교한다.

- 현빈의 `src/models/*.py`, `notebooks/compare_models.ipynb`는 건드리지 않음
- 새로 추가한 파일: `src/models/svd.py`, `src/models/mab.py` (+ `src/models/__init__.py`의
  `MODELS` 딕셔너리에 두 모델 등록 — 기존 4개 항목은 그대로 두고 추가만 함)

## 1. SVD — 편향 포함 행렬분해

논문 2장 1절 공식을 그대로 구현 (`src/models/svd.py`):

$$\hat{r}_{u,i} = \mu + b_u + b_i + p_u \cdot q_i$$

`ctx["matrix"]`(유저×게임, 로그 플레이타임)의 **관측된 값만** SGD로 학습. User-CF/Item-CF가
직접 이웃을 못 찾는 희소 게임(1인 보유 등)에서도 latent factor로 취향을 일반화할 수 있는 것이
SVD의 강점이다.

## 2. MAB — Epsilon-Greedy (논문 3장 "분류-MAB")

논문에서 가장 성능이 좋았던 조합을 그대로 재현 (`src/models/mab.py`):

1. 유저의 태그 취향(`ctx["taste"]`) 중 **rating이 가장 높은 태그 1개**로 후보 게임을 추린다
   (논문 표3: 태그 1개만 쓴 방법이 4가지 중 NDCG@10 최고 — 0.4617)
2. 후보가 20개를 넘으면 인기도(게임 이용 유저수) 상위 20개로 축소, 부족하면 인기도로 보충
   (논문은 "유저 본인 이용시간"으로 줄였지만, 여긴 유저가 안 해본 게임을 추천하는 과제라 유저
   본인 이용시간이 없음 → 인기도로 대체)
3. 후보 20개를 arm으로 **Epsilon-Greedy MAB**(논문 표4에서 3개 MAB 중 최고: 0.4653)를
   시뮬레이션. 오프라인 평가라 실제 유저 반응이 없으므로, reward는 "게임 이용 유저 비율"을
   확률로 쓰는 베르누이 시행으로 대체.
4. 200회 pull 후 추정 가치(Q) 순으로 점수화.

In [1]:
import sys
sys.path.insert(0, "../src")   # 노트북이 notebooks/ 에 있을 때
import data, evaluation
from models import MODELS

print("등록된 모델:", list(MODELS.keys()))

interactions, game, ug, ut = data.load_raw("../data/processed")
interactions = interactions[interactions["steamid"].isin(set(ug["steamid"]))]
print("상호작용:", len(interactions), "| 유저:", interactions["steamid"].nunique(),
      "| 게임:", interactions["appid"].nunique())

등록된 모델: ['popularity', 'content', 'user_cf', 'item_cf', 'svd', 'mab']
상호작용: 23297 | 유저: 779 | 게임: 3256


## 3. train/test 분할 + ctx 준비

현빈의 실험과 동일한 조건(hold-out 30%, seed=42)으로 맞춰서 공정하게 비교한다.

In [2]:
train, test = evaluation.train_test_split(interactions, test_ratio=0.3, min_games=8, seed=42)
print("학습 상호작용:", len(train), "| 평가 유저:", len(test))

ctx = data.build_context(train, game, ug, ut)
print("유저×게임 행렬:", ctx["matrix"].shape)

학습 상호작용: 16312 | 평가 유저: 779


유저×게임 행렬: (779, 2703)


## 4. 6개 모델 NDCG 비교

SVD는 `recommend()`가 처음 호출될 때 한 번만 학습되고(`ctx["_svd_pred"]`에 캐싱), 이후 유저별
호출은 캐싱된 예측 행렬에서 조회만 한다.

In [3]:
import time

t0 = time.time()
table = evaluation.evaluate(MODELS, ctx, test, k=10)
print(f"평가 소요 시간: {time.time() - t0:.1f}s")
table.round(4)

평가 소요 시간: 12.0s


,NDCG@10,Recall@10
item_cf,0.2150,0.1704
user_cf,0.1905,0.1441
popularity,0.1543,0.1248
content,0.0883,0.0703
mab,0.0794,0.0574
svd,0.0646,0.0540


## 5. Random 기준선과 비교

절대값이 낮아 보일 수 있어 random 기준선을 하나 더 붙인다 (`docs/recommendation_models.md` 4절
"절대값이 아니라 인기도 baseline 대비로 판단" 원칙과 같은 맥락).

In [4]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)

def _random_scores(user, ctx):
    return pd.Series(rng.random(len(ctx["all_games"])), index=ctx["all_games"])

rand_ndcgs, rand_recs = [], []
for sid, rel in test.items():
    ow = ctx["owned"].get(sid, set())
    scores = _random_scores(sid, ctx)
    rand_ndcgs.append(evaluation.ndcg_at_k(scores, rel, ow, k=10))
    rand_recs.append(evaluation.recall_at_k(scores, rel, ow, k=10))

table_with_random = pd.concat([
    table,
    pd.DataFrame({"NDCG@10": [np.mean(rand_ndcgs)], "Recall@10": [np.mean(rand_recs)]}, index=["random"]),
]).sort_values("NDCG@10", ascending=False)
table_with_random.round(4)

,NDCG@10,Recall@10
item_cf,0.2150,0.1704
user_cf,0.1905,0.1441
popularity,0.1543,0.1248
content,0.0883,0.0703
mab,0.0794,0.0574
svd,0.0646,0.0540
random,0.0016,0.0021


**결과 해석**

- SVD·MAB 둘 다 random(≈0.002)보다는 확실히 신호가 있지만, popularity·User-CF·Item-CF보다는
  낮다 — 앞선 조사(04_userCF_itemCF.ipynb)에서 확인한 것처럼 **유저 수(779명)가 적어 latent
  factor/신뢰할 수 있는 pull 횟수를 학습하기엔 데이터가 부족**하기 때문으로 보인다.
- **MAB(Epsilon-Greedy)가 SVD보다 낫다** — 논문의 핵심 결론("분류-MAB 모델이 유저-게임 SVD보다
  NDCG가 높다", 0.4653 vs 0.3227)과 같은 방향의 결과다. 태그 1개로 후보를 좁힌 뒤 인기도 기반
  reward로 MAB를 돌리는 것이, SVD의 순수 latent factor보다 이 데이터 규모에서 더 안정적으로
  작동한다는 뜻.
- 다만 현빈의 User-CF/Item-CF(가중합, 정규화 없음)에는 못 미친다 — 그쪽은 "유사 유저들 사이의
  인기도"를 직접 반영하는 형태라, 이 작은 데이터셋에서는 더 강한 신호로 작동하는 것으로 보인다
  (자세한 원인 분석은 별도 조사 완료: 정규화(가중평균) 유무가 결정적이었음).

## 7. Item-CF + Content 하이브리드

가장 성능이 좋았던 Item-CF(0.215)에 Content(0.088)를 얼마나 섞으면 좋을지 실험한다. 둘은
서로 다른 재료(Item-CF=플레이 co-occurrence, Content=장르·태그 메타데이터)에서 나온 점수라
어느 정도 보완 효과를 기대할 수 있다.

`item_cf.py`/`content.py`는 건드리지 않고, 두 모델의 출력 점수를 유저별로 min-max 정규화한 뒤
$score = \alpha \cdot item\_cf + (1-\alpha) \cdot content$ 로 가중합해서 α를 홀드아웃 NDCG@10
기준으로 스윕한다.

In [5]:
from models import item_cf, content


def _normalize(scores, owned):
    s = scores.drop(index=[a for a in owned if a in scores.index], errors="ignore")
    if s.max() == s.min():
        return s * 0.0
    return (s - s.min()) / (s.max() - s.min())


# 모델별 점수를 유저마다 한 번만 계산해 캐싱 (alpha 스윕 때 재사용)
item_scores, content_scores = {}, {}
for sid in test.keys():
    owned = ctx["owned"].get(sid, set())
    item_scores[sid] = _normalize(item_cf.recommend(sid, ctx), owned)
    content_scores[sid] = _normalize(content.recommend(sid, ctx), owned)

hybrid_results = []
for alpha in [1.0, 0.95, 0.9, 0.85, 0.8, 0.75, 0.7, 0.6, 0.5, 0.3, 0.0]:
    ndcgs, recs = [], []
    for sid, rel in test.items():
        owned = ctx["owned"].get(sid, set())
        combo = alpha * item_scores[sid] + (1 - alpha) * content_scores[sid]
        ndcgs.append(evaluation.ndcg_at_k(combo, rel, owned, k=10))
        recs.append(evaluation.recall_at_k(combo, rel, owned, k=10))
    hybrid_results.append({"alpha (item_cf 비중)": alpha, "NDCG@10": np.mean(ndcgs), "Recall@10": np.mean(recs)})

hybrid_df = pd.DataFrame(hybrid_results).sort_values("NDCG@10", ascending=False).reset_index(drop=True)
hybrid_df.round(4)

,alpha (item_cf 비중),NDCG@10,Recall@10
0,0.85,0.2180,0.1727
1,0.90,0.2175,0.1718
2,0.80,0.2173,0.1719
3,0.95,0.2164,0.1708
4,1.00,0.2150,0.1704
5,0.70,0.2147,0.1701
6,0.75,0.2146,0.1691
7,0.60,0.2119,0.1685
8,0.50,0.2057,0.1639
9,0.30,0.1794,0.1454


**결과**: α=0.85(Item-CF 85% + Content 15%) 근방이 최고점이고, Item-CF 단독(α=1.0, 0.2150)보다
소폭 높다(약 +1.4%). 큰 개선은 아니지만 일관되게 α=0.75~0.95 구간에서 Item-CF 단독을 웃돈다 —
Content가 완전히 겹치는 신호는 아니고, 약간의 보완 효과가 있다는 뜻이다. 다만 이미 Item-CF가
워낙 강해서(β=0 근처, 즉 Content 비중이 커질수록 급격히 나빠짐 — 아래 β>0.3 구간 참고),
"하이브리드로 큰 도약"이라기보다는 "미세 조정" 수준의 개선이다.

In [6]:
best_hybrid = hybrid_df.iloc[0]
best_alpha = best_hybrid["alpha (item_cf 비중)"]
print(f"최적 alpha={best_alpha}  NDCG@10={best_hybrid['NDCG@10']:.4f} "
      f"(Item-CF 단독 대비 {(best_hybrid['NDCG@10']/table.loc['item_cf','NDCG@10'] - 1)*100:+.1f}%)")

table_final = pd.concat([
    table_with_random,
    pd.DataFrame({"NDCG@10": [best_hybrid["NDCG@10"]], "Recall@10": [best_hybrid["Recall@10"]]},
                 index=[f"item_cf+content (alpha={best_alpha})"]),
]).sort_values("NDCG@10", ascending=False)
table_final.round(4)

최적 alpha=0.85  NDCG@10=0.2180 (Item-CF 단독 대비 +1.4%)


,NDCG@10,Recall@10
item_cf+content (alpha=0.85),0.2180,0.1727
item_cf,0.2150,0.1704
user_cf,0.1905,0.1441
popularity,0.1543,0.1248
content,0.0883,0.0703
mab,0.0794,0.0574
svd,0.0646,0.0540
random,0.0016,0.0021


## 6. 결과 저장

In [7]:
import os
os.makedirs("../results", exist_ok=True)
table_final.to_csv("../results/model_comparison_svd_mab.csv", encoding="utf-8-sig")
print("저장: results/model_comparison_svd_mab.csv")

저장: results/model_comparison_svd_mab.csv
